In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score,f1_score,recall_score
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler,OneHotEncoder

In [1]:
import pandas as pd
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt
path = "C:/Users/admin/Desktop/self-healing-ml/self-healing-model/data/raw/online_shoppers_intention.csv"
train_path = "C:/Users/admin/Desktop/self-healing-ml/self-healing-model/data/raw/online_shoppers_intention_train.csv"
test_path = "C:/Users/admin/Desktop/self-healing-ml/self-healing-model/data/raw/online_shoppers_intention_test.csv"

In [2]:
df = pd.read_csv(path)
train = pd.read_csv(train_path)
test = pd.read_csv(test_path)
df = pd.get_dummies(df)


In [3]:
train.columns

Index(['Administrative', 'Administrative_Duration', 'Informational',
       'Informational_Duration', 'ProductRelated', 'ProductRelated_Duration',
       'BounceRates', 'ExitRates', 'PageValues', 'SpecialDay', 'Month',
       'OperatingSystems', 'Browser', 'Region', 'TrafficType', 'VisitorType',
       'Weekend', 'Revenue'],
      dtype='object')

In [8]:
train.dtypes.to_dict()

{'Administrative': dtype('int64'),
 'Administrative_Duration': dtype('float64'),
 'Informational': dtype('int64'),
 'Informational_Duration': dtype('float64'),
 'ProductRelated': dtype('int64'),
 'ProductRelated_Duration': dtype('float64'),
 'BounceRates': dtype('float64'),
 'ExitRates': dtype('float64'),
 'PageValues': dtype('float64'),
 'SpecialDay': dtype('float64'),
 'Month': dtype('O'),
 'OperatingSystems': dtype('int64'),
 'Browser': dtype('int64'),
 'Region': dtype('int64'),
 'TrafficType': dtype('int64'),
 'VisitorType': dtype('O'),
 'Weekend': dtype('bool'),
 'Revenue': dtype('bool')}

In [9]:
for key,val in train.dtypes.to_dict().items():
    print(f"{key}:{val}")

Administrative:int64
Administrative_Duration:float64
Informational:int64
Informational_Duration:float64
ProductRelated:int64
ProductRelated_Duration:float64
BounceRates:float64
ExitRates:float64
PageValues:float64
SpecialDay:float64
Month:object
OperatingSystems:int64
Browser:int64
Region:int64
TrafficType:int64
VisitorType:object
Weekend:bool
Revenue:bool


## Base model

In [ ]:
X = df.drop("Revenue", axis=1)
y = df["Revenue"]

X_train,X_test,y_train,y_test = train_test_split(X,y,random_state=42,test_size=0.2)

In [ ]:
lr = LogisticRegression()
lr.fit(X_train,y_train)

accuracy_score(y_test,lr.predict(X_test))

## After transformation

In [ ]:
num_cols = df.select_dtypes(include=["int64", "float64"]).columns
before = {}
for col in num_cols:
    plt.figure(figsize=(4,3))
    sns.kdeplot(df[col])
    plt.title(col)
    plt.show()
    before[col] = df[col].skew()
    print(f"{col}:- {df[col].skew()}")

In [ ]:
num_cols = df.select_dtypes(include=["int64", "float64"]).columns
skewed_cols = df[num_cols].skew().sort_values(ascending=False)

high_skew = skewed_cols[skewed_cols > 1].index

for col in high_skew:
    df[col] = np.log1p(df[col])

In [ ]:
num_cols = df.select_dtypes(include=["int64", "float64"]).columns
after = {}
for col in num_cols:
    plt.figure(figsize=(6,4))
    sns.kdeplot(df[col])
    plt.title(col)
    plt.show()
    after[col] = df[col].skew()
    print(f"{col}:- {df[col].skew()}")

In [ ]:
skew = pd.DataFrame({"column": num_cols,
    "before": [before[col] for col in num_cols],
    "after": [after[col] for col in num_cols]})

In [ ]:
skew

In [ ]:
X = df.drop("Revenue", axis=1)
y = df["Revenue"]

X_train,X_test,y_train,y_test = train_test_split(X,y,random_state=42,test_size=0.2)

In [ ]:
lr1 = LogisticRegression(max_iter=1000)

lr1.fit(X_train,y_train)

accuracy_score(lr1.predict(X_test),y_test)

In [ ]:
train.info()

In [ ]:
num_cols = train.select_dtypes(include=["int64", "float64"]).columns


In [ ]:
object_cols = train.select_dtypes(include=["object"]).columns


In [ ]:
object_cols

In [23]:
def column_transformation(data: pd.DataFrame, median_val) -> pd.DataFrame:

    df = data.copy()

    # Total time = strong engagement signal
    df["EngagementScore"] = (
                                df["ProductRelated_Duration"] +
                                df["Informational_Duration"] +
                                df["Administrative_Duration"]
                            )
    
    # Focus on product vs other pages
    df["InteractionRate"] = df["ProductRelated"] / (df["Administrative"] + 1)

    # Detect poor sessions
    df["BounceExitRatio"] = df["BounceRates"] / (df["ExitRates"] + 1e-5)

    # Returning users buy more
    df["IsReturning"] = (df["VisitorType"] == "Returning_Visitor").astype(int)

    # Captures serious buyers
    if median_val is not None:
        df["HighIntent"] = (df["ProductRelated_Duration"] > median_val).astype(int)
    
    # Deep engagement indicator
    df["AvgTimePerPage"] = df["ProductRelated_Duration"] / (df["ProductRelated"] + 1)

    # Business + behavior together
    df["UserQuality"] = (
                            df["PageValues"] * df["ProductRelated_Duration"]
                        )
    df = df.round(2)

    return df


def column_scaling(train:pd.DataFrame,test:pd.DataFrame):

    # fatch the column names
    num_cols = train.select_dtypes(include=["int64", "float64"]).columns
    object_cols = train.select_dtypes(include=["object"]).columns

    # making the transformer 
    transformer = ColumnTransformer(transformers=[
        ("StandardScaler",StandardScaler(),num_cols),
        ("OHE",OneHotEncoder(handle_unknown="ignore", sparse_output=False),object_cols)
    ],
    remainder="passthrough")

    #scaling the column
    train_arr = transformer.fit_transform(train)
    test_arr = transformer.transform(test)

    feature_names = transformer.get_feature_names_out()

    train = pd.DataFrame(train_arr, columns=feature_names)
    test = pd.DataFrame(test_arr, columns=feature_names)

    return train,test

In [24]:
median_val = train["ProductRelated_Duration"].median()
train_data = column_transformation(train,median_val)
test_data = column_transformation(test,median_val)

In [26]:
train,test = column_scaling(train_data,test_data)


In [ ]:
lr2 = LogisticRegression()
lr2.fit(train.drop("rev"))

In [29]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9864 entries, 0 to 9863
Data columns (total 36 columns):
 #   Column                                   Non-Null Count  Dtype  
---  ------                                   --------------  -----  
 0   StandardScaler__Administrative           9864 non-null   float64
 1   StandardScaler__Administrative_Duration  9864 non-null   float64
 2   StandardScaler__Informational            9864 non-null   float64
 3   StandardScaler__Informational_Duration   9864 non-null   float64
 4   StandardScaler__ProductRelated           9864 non-null   float64
 5   StandardScaler__ProductRelated_Duration  9864 non-null   float64
 6   StandardScaler__BounceRates              9864 non-null   float64
 7   StandardScaler__ExitRates                9864 non-null   float64
 8   StandardScaler__PageValues               9864 non-null   float64
 9   StandardScaler__SpecialDay               9864 non-null   float64
 10  StandardScaler__OperatingSystems         9864 no